In [150]:
import pandas as pd

df = pd.read_csv("dataset/new.csv", encoding="latin1")

print(df.head())
print(df.columns)


   doc ID                    title  \
0     1.0  Artificial intelligence   
1     NaN                      NaN   
2     NaN                      NaN   
3     NaN                      NaN   
4     NaN                      NaN   

                                                text  
0  Artificial intelligence (AI) is the capability...  
1  associated with human intelligence, such as le...  
2  perception, and decision-making. It is a field...  
3  and studies methods and software that enable m...  
4  use learning and intelligence to take actions ...  
Index(['doc ID', 'title', 'text'], dtype='str')


In [151]:
import pandas as pd

#  Read CSV
df = pd.read_csv("dataset/new.csv", encoding="latin1")

#  Rename column
df = df.rename(columns={"doc ID": "doc_id"})

#  Forward fill IDs and titles
df["doc_id"] = df["doc_id"].ffill()
df["title"] = df["title"].ffill()

#  SAFE text cleaning BEFORE groupby
df["text"] = df["text"].apply(
    lambda x: str(x) if pd.notna(x) else ""
)

# SAFE groupby + join (FLOATS REMOVED HERE)
df_clean = (
    df.groupby(["doc_id", "title"], as_index=False)
      .agg({
          "text": lambda x: " ".join(
              [t for t in x if isinstance(t, str)]
          )
      })
)

df_clean["doc_id"] = df_clean["doc_id"].astype(int)
# df_clean = df_clean.reset_index(drop=True)


print(df_clean.head(15))
print("Total documents:", len(df_clean))


    doc_id                                     title  \
0        1                   Artificial intelligence   
1        2                          Machine learning   
2        3        Generative artificial intelligence   
3        4               Natural language processing   
4        5                             Deep learning   
5        6                     Experiential learning   
6        7                            Neural network   
7        8                     Information retrieval   
8        9                           Computer vision   
9       10                    Emotional intelligence   
10      11                        Internet of things   
11      12                         Neuro-symbolic AI   
12      13  Constructivism (philosophy of education)   
13      14                              Data science   
14      15                                  Big data   

                                                 text  
0   Artificial intelligence (AI) is the capabil

The corpus contains 15 documents with an average length of approximately 661 words.
The longest document contains 1083 words, justifying a chunk size of 200–300 words
for effective retrieval and explainability.


In [152]:
# Add word count column
df_clean["word_count"] = df_clean["text"].apply(lambda x: len(str(x).split()))

# Stats
avg_length = df_clean["word_count"].mean()
max_length = df_clean["word_count"].max()

print("Average document length (words):", int(avg_length))
print("Maximum document length (words):", max_length)


Average document length (words): 661
Maximum document length (words): 1083


Document-level analysis shows word counts ranging from ~350 to ~1080 words.
This variability supports a chunking strategy of 200–300 words, resulting in
approximately 2–5 chunks per document.


In [153]:
# Word count per document
df_clean["word_length"] = df_clean["text"].apply(lambda x: len(x.split()))

# Character count per document
df_clean["char_length"] = df_clean["text"].apply(lambda x: len(x))

# Display result
df_clean[["doc_id", "title", "word_length", "char_length"]]


,doc_id,title,word_length,char_length
0,1,Artificial intelligence,402,2804
1,2,Machine learning,699,4853
2,3,Generative artificial intelligence,421,2826
3,4,Natural language processing,1043,7119
4,5,Deep learning,962,6526
5,6,Experiential learning,658,4450
6,7,Neural network,1083,7241
7,8,Information retrieval,359,2282
8,9,Computer vision,789,5389
9,10,Emotional intelligence,746,5079


Keyword frequency analysis (after stopword removal and minimum length filtering)
shows dominant terms such as learning, neural, machine, intelligence, and systems,
confirming the coherence of the AI-focused document corpus.


In [154]:
from collections import Counter
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Combine all text
all_text = " ".join(df_clean["text"].astype(str)).lower()

# Extract words (only alphabets)
words = re.findall(r"\b[a-zA-Z]+\b", all_text)

# Filter: length >= 5 and remove stopwords
filtered_words = [
    w for w in words
    if len(w) >= 5 and w not in ENGLISH_STOP_WORDS
]

# Count frequency
word_freq = Counter(filtered_words)

# Top 20 most used words
print("Top 20 most used words (length ≥ 5):")
for word, count in word_freq.most_common(20):
    print(word, ":", count)


Top 20 most used words (length ≥ 5):
learning : 139
neural : 50
information : 47
computer : 45
networks : 45
science : 39
machine : 39
intelligence : 36
systems : 33
knowledge : 30
artificial : 28
language : 28
model : 28
research : 26
models : 26
network : 26
processing : 25
field : 24
layer : 24
methods : 22


In [155]:
import pandas as pd
import json

#  Read CSV
df = pd.read_csv("dataset/new.csv", encoding="latin1")

#  Rename column (if needed)
df = df.rename(columns={"doc ID": "doc_id"})

#  Forward fill doc_id & title
df["doc_id"] = df["doc_id"].ffill()
df["title"] = df["title"].ffill()

#  Clean text safely
df["text"] = df["text"].apply(lambda x: str(x) if pd.notna(x) else "")

#  MERGE rows → ONE document
df_clean = (
    df.groupby(["doc_id", "title"], as_index=False)
      .agg({
          "text": lambda x: " ".join([t for t in x if isinstance(t, str)])
      })
)

#  Convert to JSON-safe dict
records = [
    {
        "doc_id": int(row["doc_id"]),
        "title": row["title"],
        "text": row["text"]
    }
    for _, row in df_clean.iterrows()
]

#  Save JSON (NO NaN, NO ERRORS)
with open("raw_docs.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print("Total documents:", len(records))


Total documents: 15


In [156]:
def find_weird_chars(text):
    return sorted(set(
        ch for ch in text
        if not ch.isprintable() or ord(ch) > 127
    ))

# Check first few documents
for i in range(min(15, len(df_clean))):
    weird = find_weird_chars(df_clean.loc[i, "text"])
    print(f"Doc {i+1} weird chars:", weird[:20])


Doc 1 weird chars: ['\n', '\r']
Doc 2 weird chars: []
Doc 3 weird chars: []
Doc 4 weird chars: ['\x9a', 'á']
Doc 5 weird chars: []
Doc 6 weird chars: []
Doc 7 weird chars: []
Doc 8 weird chars: []
Doc 9 weird chars: ['\n', '\r']
Doc 10 weird chars: ['\x96']
Doc 11 weird chars: ['\x97']
Doc 12 weird chars: []
Doc 13 weird chars: ['\x96']
Doc 14 weird chars: []
Doc 15 weird chars: ['\x80', '×']


In [157]:
import re
import unicodedata

def final_text_cleaner(text):
    if not isinstance(text, str):
        return ""

    # Normalize unicode (á → a, smart chars fixed)
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")

    # Remove control / extended ASCII chars
    text = re.sub(r"[\x00-\x1F\x7F-\x9F]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()


df_clean["text"] = df_clean["text"].apply(final_text_cleaner)
print("Final text cleaning applied")


Final text cleaning applied


In [158]:
for i in range(min(15, len(df_clean))):
    weird = find_weird_chars(df_clean.loc[i, "text"])
    print(f"Doc {i+1} weird chars:", weird)


Doc 1 weird chars: []
Doc 2 weird chars: []
Doc 3 weird chars: []
Doc 4 weird chars: []
Doc 5 weird chars: []
Doc 6 weird chars: []
Doc 7 weird chars: []
Doc 8 weird chars: []
Doc 9 weird chars: []
Doc 10 weird chars: []
Doc 11 weird chars: []
Doc 12 weird chars: []
Doc 13 weird chars: []
Doc 14 weird chars: []
Doc 15 weird chars: []


In [159]:
import re

for i in range(min(15, len(df_clean))):
    refs = re.findall(r"\[\d+\]", df_clean.loc[i, "text"])
    print(f"Doc {i+1} references found:", refs[:5])


Doc 1 references found: []
Doc 2 references found: []
Doc 3 references found: []
Doc 4 references found: []
Doc 5 references found: []
Doc 6 references found: []
Doc 7 references found: []
Doc 8 references found: []
Doc 9 references found: []
Doc 10 references found: []
Doc 11 references found: []
Doc 12 references found: []
Doc 13 references found: []
Doc 14 references found: []
Doc 15 references found: []


In [160]:
for i in range(min(15, len(df_clean))):
    text = df_clean.loc[i, "text"]
    print(f"Doc {i+1} newline count:", text.count("\n"))


Doc 1 newline count: 0
Doc 2 newline count: 0
Doc 3 newline count: 0
Doc 4 newline count: 0
Doc 5 newline count: 0
Doc 6 newline count: 0
Doc 7 newline count: 0
Doc 8 newline count: 0
Doc 9 newline count: 0
Doc 10 newline count: 0
Doc 11 newline count: 0
Doc 12 newline count: 0
Doc 13 newline count: 0
Doc 14 newline count: 0
Doc 15 newline count: 0


In [161]:
html_tokens = ["&nbsp;", "&amp;", "&lt;", "&gt;"]

for token in html_tokens:
    found = any(token in t for t in df_clean["text"])
    print(token, "found:", found)


&nbsp; found: False
&amp; found: False
&lt; found: False
&gt; found: False


In [162]:
summary = {
    "has_references": any(re.search(r"\[\d+\]", t) for t in df_clean["text"]),
    "has_nonbreaking_space": any("\xa0" in t for t in df_clean["text"]),
    "has_zero_width_space": any("\u200b" in t for t in df_clean["text"]),
    "has_html_entities": any("&" in t for t in df_clean["text"]),
    "has_newlines": any("\n" in t for t in df_clean["text"]),
}

print(summary)


{'has_references': False, 'has_nonbreaking_space': False, 'has_zero_width_space': False, 'has_html_entities': True, 'has_newlines': False}


In [163]:
import re

entities_found = set()

for text in df_clean["text"]:
    matches = re.findall(r"&[a-zA-Z]+;", text)
    entities_found.update(matches)

print("HTML entities found:", entities_found)


HTML entities found: set()


In [164]:
import re
import html

def remove_html_entities_precise(text):
    if not isinstance(text, str):
        return ""

    # Convert HTML entities to normal characters
    text = html.unescape(text)

    # Remove leftover entities if any (&word;)
    text = re.sub(r"&[a-zA-Z]+;", "", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()


# Apply fix
df_clean["text"] = df_clean["text"].apply(remove_html_entities_precise)

print("✅ HTML entities removed precisely")


✅ HTML entities removed precisely


In [165]:
summary = {
    "has_references": any(re.search(r"\[\d+\]", t) for t in df_clean["text"]),
    "has_nonbreaking_space": any("\xa0" in t for t in df_clean["text"]),
    "has_zero_width_space": any("\u200b" in t for t in df_clean["text"]),
    "has_html_entities": any(re.search(r"&[a-zA-Z]+;", t) for t in df_clean["text"]),
    "has_newlines": any("\n" in t for t in df_clean["text"]),
}

print(summary)


{'has_references': False, 'has_nonbreaking_space': False, 'has_zero_width_space': False, 'has_html_entities': False, 'has_newlines': False}


In [173]:
import os

os.rename("raw_docs.json", "raw_cleaned.json")
print("✅ File renamed to documents.json")


✅ File renamed to documents.json
